# Automatické tréningy pre rôzne n_input a horizonty (DST+1 … DST+6)

Tento notebook načíta dáta **iba raz** a potom postupne vytvorí a natrénuje samostatný model pre každú kombináciu:
- `n_input` ∈ {6, 12, 18, 24, 30, 36, 42, 48}
- `y_col` = `DST+1` … `DST+6`

Model/weighty sa ukladajú do `models/` a súhrn metrík do `results/summary.csv`.


In [1]:
import os
import numpy as np
import pandas as pd

from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from keras.models import Model
from keras.layers import Dense, Input, LSTM, Flatten, TimeDistributed, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Reprodukovateľnosť (voliteľné)
SEED = 42
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)


2026-04-10 14:13:48.428756: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import os; 
print(os.getcwd())

/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/3_modelovanie/train automate


In [3]:
# =========================
# Nastavenia
# =========================

# Cesty k datasetom (ponechané ako v pôvodnom notebooku)
BASE_PATH = "/home/jovyan/data/lightning/TereziaD/DP_pokracovanieBP/BP_Drengubiakova_Terezia/0_datasety/"
file_path_train = BASE_PATH + "train_omni.csv"
file_path_test  = BASE_PATH + "test_omni.csv"


# Tréningové nastavenia
BATCH_SIZE = 256
EPOCHS = 200
PATIENCE = 25  # early stopping na val_mae

# Kombinácie na tréning
#N_INPUT_LIST = [36, 42, 48]
N_INPUT_LIST = [6, 12, 18, 24, 30, 36, 42, 48]
HORIZONS = [1]

# Výstupné priečinky
MODELS_DIR = "models"
RESULTS_DIR = "results"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [4]:
# =========================
# Načítanie dát (iba raz)
# =========================
train_raw = pd.read_csv(file_path_train)
test_raw  = pd.read_csv(file_path_test)

# Skontroluj, aké DST+ stĺpce existujú (pomôže odhaliť preklepy v názvoch)
dst_cols = [c for c in train_raw.columns if c.startswith("DST+")]
print("DST+ stĺpce v train:", dst_cols)
print("DST+ stĺpce v test :", [c for c in test_raw.columns if c.startswith("DST+")])

# čas
if "time1" in train_raw.columns:
    train_raw["time1"] = pd.to_datetime(train_raw["time1"])
if "time1" in test_raw.columns:
    test_raw["time1"] = pd.to_datetime(test_raw["time1"])


DST+ stĺpce v train: ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']
DST+ stĺpce v test : ['DST+1', 'DST+2', 'DST+3', 'DST+4', 'DST+5', 'DST+6']


In [5]:
train_raw

,Unnamed: 0.1,Unnamed: 0,time1,bz_gsm,v,DST,DST+1,DST+2,DST+3,DST+4,DST+5,DST+6
0,0,0,1963-01-01 00:30:00+00:00,-0.2,285.0,-6,-6,-5.0,-5.0,-3.0,-3.0,-6.0
1,1,1,1963-01-01 01:30:00+00:00,-0.2,285.0,-5,-5,-5.0,-3.0,-3.0,-6.0,-8.0
2,2,2,1963-01-01 02:30:00+00:00,-0.2,285.0,-5,-5,-3.0,-3.0,-6.0,-8.0,-9.0
3,3,3,1963-01-01 03:30:00+00:00,-0.2,285.0,-3,-3,-3.0,-6.0,-8.0,-9.0,-6.0
4,4,4,1963-01-01 04:30:00+00:00,-0.2,285.0,-3,-3,-6.0,-8.0,-9.0,-6.0,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
286690,286690,286690,1995-09-15 10:30:00+00:00,-5.6,429.0,-52,-52,-48.0,-40.0,-39.0,-45.0,-48.0
286691,286691,286691,1995-09-15 11:30:00+00:00,-2.9,435.0,-48,-48,-40.0,-39.0,-45.0,-48.0,-45.0
286692,286692,286692,1995-09-15 12:30:00+00:00,-7.8,440.0,-40,-40,-39.0,-45.0,-48.0,-45.0,-41.0
286693,286693,286693,1995-09-15 13:30:00+00:00,-5.6,440.0,-39,-39,-45.0,-48.0,-45.0,-41.0,-41.0


In [6]:
def build_model(n_input: int, n_features: int) -> keras.Model:
    """LSTM model pre multivariačný vstup."""
    inputs = Input(shape=(n_input, n_features))

    x = Bidirectional(
        LSTM(128, return_sequences=True, dropout=0.1, recurrent_dropout=0.1)
    )(inputs)
    x = LSTM(128, return_sequences=True)(x)
    x = TimeDistributed(Dense(1, activation="linear"))(x)
    x = Flatten()(x)
    outputs = Dense(1, activation="linear")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss="mse", optimizer="adam", metrics=["mae"])
    return model


def make_splits(train_df: pd.DataFrame, test_df: pd.DataFrame, y_col: str, predictors=None):
    """Pripraví train/valid/test dáta pre daný y_col (bez NaN)."""

    if predictors is None:
        predictors = ["DST", "bz_gsm"]   # alebo ["DST", "v"], podľa toho čo chceš reálne používať

    features = predictors + [y_col]

    train = train_df[features].copy()
    test = test_df[features].copy()

    # odstránenie NaN
    train = train.dropna().reset_index(drop=True)
    test = test.dropna().reset_index(drop=True)

    # časový split train/valid
    valid_size = int(len(train) * 0.2)

    if valid_size == 0:
        raise ValueError(f"Príliš málo dát po dropna pre {y_col}")

    valid = train.iloc[-valid_size:, :].copy()
    train = train.iloc[:-valid_size, :].copy()

    # X = 2 features
    X_train = train[predictors].values
    y_train = train[y_col].values

    X_val = valid[predictors].values
    y_val = valid[y_col].values

    X_test = test[predictors].values
    y_test = test[y_col].values

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), predictors

In [7]:
def train_one(y_col: str, n_input: int):
    predictors = ["DST", "bz_gsm"]   

    (X_train, y_train), (X_val, y_val), (X_test, y_test), predictors = make_splits(
        train_raw, test_raw, y_col, predictors=predictors
    )

    train_gen = TimeseriesGenerator(X_train, y_train, length=n_input, batch_size=BATCH_SIZE)
    val_gen   = TimeseriesGenerator(X_val, y_val, length=n_input, batch_size=BATCH_SIZE)
    test_gen  = TimeseriesGenerator(X_test, y_test, length=n_input, batch_size=BATCH_SIZE)

    if len(train_gen) == 0 or len(val_gen) == 0:
        print(f"[SKIP] {y_col}, n_input={n_input} – prázdny generátor")
        return None

    n_features = len(predictors)
    model = build_model(n_input, n_features)

    print("Predictors:", predictors)
    print("X_train shape:", X_train.shape)
    print("First batch X shape:", train_gen[0][0].shape)
    print("Model input shape:", model.input_shape)

    model_path = os.path.join(MODELS_DIR, f"{y_col}_{n_input}H_BZ.keras")
    checkpoint = ModelCheckpoint(model_path, monitor="val_mae", verbose=1, save_best_only=True, mode="min")
    early = EarlyStopping(monitor="val_mae", mode="min", patience=PATIENCE, restore_best_weights=True)
    callbacks = [checkpoint, early]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        verbose=1,
        callbacks=callbacks
    )

    test_loss, test_mae = model.evaluate(test_gen, verbose=0)

    hist_df = pd.DataFrame(history.history)
    hist_csv = os.path.join(RESULTS_DIR, f"history_{y_col}_{n_input}H_BZ.csv")
    hist_df.to_csv(hist_csv, index=False)

    return {
        "y_col": y_col,
        "horizon_hours": int(y_col.split("+")[1]),
        "n_input": n_input,
        "best_val_mae": float(np.min(hist_df["val_mae"])) if "val_mae" in hist_df else np.nan,
        "best_val_loss": float(np.min(hist_df["val_loss"])) if "val_loss" in hist_df else np.nan,
        "test_mae": float(test_mae),
        "test_loss": float(test_loss),
        "model_path": model_path,
        "history_path": hist_csv,
        "epochs_ran": int(len(hist_df)),
    }

In [8]:
# =========================
# Spustenie všetkých tréningov
# =========================
summary_rows = []

for h in HORIZONS:
    y_col = f"DST+{h}"

    # ochrana, ak by stĺpec neexistoval
    if y_col not in train_raw.columns or y_col not in test_raw.columns:
        print(f"[SKIP] {y_col} neexistuje v datasete.")
        continue

    for n_input in N_INPUT_LIST:
        print("\n" + "="*80)
        print(f"Trénujem: y_col={y_col}, n_input={n_input}")
        print("="*80)

        row = train_one(y_col, n_input)
        if row is not None:
            summary_rows.append(row)


summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(RESULTS_DIR, "summary_1_BZ.csv")
summary.to_csv(summary_path, index=False)

summary.sort_values(["horizon_hours", "n_input"]).head(20), summary_path



Trénujem: y_col=DST+1, n_input=6
Predictors: ['DST', 'bz_gsm']
X_train shape: (229356, 2)
First batch X shape: (256, 6, 2)
Model input shape: (None, 6, 2)
Epoch 1/200


/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 231.5331 - mae: 7.4930
Epoch 1: val_mae improved from inf to 4.60289, saving model to models/DST+1_6H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 64s 66ms/step - loss: 231.4059 - mae: 7.4902 - val_loss: 116.9679 - val_mae: 4.6029
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - loss: 45.5852 - mae: 3.4471
Epoch 2: val_mae improved from 4.60289 to 4.04267, saving model to models/DST+1_6H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 78s 87ms/step - loss: 45.5735 - mae: 3.4468 - val_loss: 71.0937 - val_mae: 4.0427
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 24.3883 - mae: 3.0022
Epoch 3: val_mae did not improve from 4.04267
896/896 ━━━━━━━━━━━━━━━━━━━━ 76s 84ms/step - loss: 24.3892 - mae: 3.0022 - val_loss: 65.2694 - val_mae: 4.5088
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - loss: 25.4162 - mae: 3.0536
Epoch 4: val_mae did not improve from 4.04267
896/896 ━━━━━━━━━━━━━━━━━━━━ 75s 84ms/step - loss: 25.4149 - mae: 3.

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - loss: 180.4693 - mae: 7.1474
Epoch 1: val_mae improved from inf to 4.88085, saving model to models/DST+1_12H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 133s 135ms/step - loss: 180.3863 - mae: 7.1452 - val_loss: 119.4768 - val_mae: 4.8808
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - loss: 34.2327 - mae: 3.3282
Epoch 2: val_mae did not improve from 4.88085
896/896 ━━━━━━━━━━━━━━━━━━━━ 123s 137ms/step - loss: 34.2316 - mae: 3.3281 - val_loss: 158.6156 - val_mae: 8.9035
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 46.2271 - mae: 3.6530
Epoch 3: val_mae improved from 4.88085 to 4.71733, saving model to models/DST+1_12H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 122s 136ms/step - loss: 46.2110 - mae: 3.6525 - val_loss: 70.8499 - val_mae: 4.7173
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 27.7794 - mae: 3.1242
Epoch 4: val_mae improved from 4.71733 to 3.94910, saving model to models/DST+1_12H_BZ.keras
896/896 

...

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step - loss: 183.6320 - mae: 7.8177
Epoch 1: val_mae improved from inf to 6.21600, saving model to models/DST+1_42H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 498s 540ms/step - loss: 183.5478 - mae: 7.8155 - val_loss: 143.2605 - val_mae: 6.2160
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 479ms/step - loss: 45.1560 - mae: 3.9002
Epoch 2: val_mae improved from 6.21600 to 4.18358, saving model to models/DST+1_42H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 465s 519ms/step - loss: 45.1519 - mae: 3.8999 - val_loss: 67.1197 - val_mae: 4.1836
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 428ms/step - loss: 30.3943 - mae: 3.3227
Epoch 3: val_mae improved from 4.18358 to 4.12181, saving model to models/DST+1_42H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 419s 468ms/step - loss: 30.3933 - mae: 3.3226 - val_loss: 49.6611 - val_mae: 4.1218
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 424ms/step - loss: 31.0245 - mae: 3.3625
Epoch 4: val_mae did not improve from 4.12181
896/896 ━

/opt/conda/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 846ms/step - loss: 237.8953 - mae: 9.0526
Epoch 1: val_mae improved from inf to 4.86380, saving model to models/DST+1_48H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 864s 941ms/step - loss: 237.7696 - mae: 9.0496 - val_loss: 89.7447 - val_mae: 4.8638
Epoch 2/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 841ms/step - loss: 50.6010 - mae: 4.2867
Epoch 2: val_mae did not improve from 4.86380
896/896 ━━━━━━━━━━━━━━━━━━━━ 826s 922ms/step - loss: 50.5939 - mae: 4.2864 - val_loss: 88.5928 - val_mae: 5.2595
Epoch 3/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 836ms/step - loss: 33.6947 - mae: 3.6825
Epoch 3: val_mae improved from 4.86380 to 3.91054, saving model to models/DST+1_48H_BZ.keras
896/896 ━━━━━━━━━━━━━━━━━━━━ 823s 918ms/step - loss: 33.6989 - mae: 3.6825 - val_loss: 57.0863 - val_mae: 3.9105
Epoch 4/200
896/896 ━━━━━━━━━━━━━━━━━━━━ 0s 841ms/step - loss: 36.0018 - mae: 3.6549
Epoch 4: val_mae improved from 3.91054 to 3.77953, saving model to models/DST+1_48H_BZ.keras
896/896 ━━

(   y_col  horizon_hours  n_input  best_val_mae  best_val_loss  test_mae  \
 0  DST+1              1        6      3.305070      30.328917  2.187489   
 1  DST+1              1       12      3.339955      32.893028  2.203646   
 2  DST+1              1       18      3.380495      33.479961  2.205453   
 3  DST+1              1       24      3.472144      34.684605  2.299829   
 4  DST+1              1       30      3.342152      36.148594  2.201422   
 5  DST+1              1       36      3.338700      31.704386  2.271914   
 6  DST+1              1       42      3.664238      38.188560  2.440916   
 7  DST+1              1       48      3.407975      39.691135  2.188477   
 
    test_loss                 model_path                      history_path  \
 0  11.797179   models/DST+1_6H_BZ.keras   results/history_DST+1_6H_BZ.csv   
 1  12.545404  models/DST+1_12H_BZ.keras  results/history_DST+1_12H_BZ.csv   
 2  13.429096  models/DST+1_18H_BZ.keras  results/history_DST+1_18H_BZ.csv   
 3

## Poznámky
- Ak chceš presne poradie ako si písala (najprv `DST+1` pre všetky `n_input`, potom `DST+2`, …), tak to presne robí horný loop (horizonty vonkajší, `n_input` vnútorný).
- Ak chceš opačne (pre dané `n_input` spraviť `DST+1..6`), stačí prehodiť poradie cyklov.
